In [ ]:
%pip install onnx onnxruntime skl2onnx

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import pandas as pd

# 1. Architektur gemäß Vertrag
class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 12 * 12, 128), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        # x kommt als (N, 28, 28, 1) - NHWC - von der App herein
        x = x.permute(0, 3, 1, 2)  # -> (N, 1, 28, 28)
        x = self.features(x)
        x = self.classifier(x)
        return x  # Logits (N, 10)

def main():
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")

# 2. Datensatz aus CSV laden (Angenommen: Spalte 0 ist Label, Spalten 1-784 sind Pixel)
    print("Lade CSV-Daten...")
    train_df = pd.read_csv('Datasets/mnist/mnist_train.csv')
    test_df  = pd.read_csv('Datasets/mnist/mnist_train.csv') 

    # Skalieren auf [0, 1] und Reshape auf (N, 1, 28, 28) wie bei torchvision
    x_train = torch.tensor(train_df.iloc[:, 1:].values, dtype=torch.float32) / 255.0
    y_train = torch.tensor(train_df.iloc[:, 0].values, dtype=torch.long)
    x_train = x_train.view(-1, 1, 28, 28)

    x_test = torch.tensor(test_df.iloc[:, 1:].values, dtype=torch.float32) / 255.0
    y_test = torch.tensor(test_df.iloc[:, 0].values, dtype=torch.long)
    x_test = x_test.view(-1, 1, 28, 28)

    train_ds = TensorDataset(x_train, y_train)
    test_ds  = TensorDataset(x_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    test_loader  = DataLoader(test_ds, batch_size=1000, shuffle=False)

    model = MNISTNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    # 3. Training
    epochs = 5
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            # Eingabe für das Modell explizit auf NHWC umlegen
            images = images.permute(0, 2, 3, 1).to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels_idx)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        print(f"Epoche {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f}")

    # 4. Validierung (Top-1-Accuracy >= 95%)
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.permute(0, 2, 3, 1).to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    
    if accuracy < 95.0:
        raise ValueError(f"Mindestgüte nicht erreicht! Aktuell: {accuracy:.2f}%. Abbruch des Exports.")
    print("Mindestgüte erreicht. Starte ONNX-Export...")

    # 5. ONNX Export
    os.makedirs("models", exist_ok=True)
    export_path = "models/mnist_model_v1.0.0.onnx"
    
    # Dummy-Input präzise nach App-Spezifikation (NHWC)
    model = model.to("cpu")
    dummy_input = torch.randn(1, 28, 28, 1)

    torch.onnx.export(
        model,
        dummy_input,
        export_path,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={
            "input": {0: "batch_size"},
            "output": {0: "batch_size"},
        },
        export_params=True,
        do_constant_folding=True,
        opset_version=14,
    )
    
    print(f"Export erfolgreich: {export_path}")

if __name__ == "__main__":
    main()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import os

# ==========================================
# 1. Daten-Pipeline & Preprocessing
# ==========================================
class MNISTCSVDataset(Dataset):
    def __init__(self, features, labels):
        # Spec: Normalization to [0, 1]
        self.features = features / 255.0
        # Spec: Reshape to (N, 28, 28, 1) -> (Batch, Height, Width, Channel)
        self.features = self.features.reshape(-1, 28, 28, 1)
        self.features = torch.tensor(self.features, dtype=torch.float32)
        
        # Spec: Convert labels to categorical (one-hot encoding)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.labels_one_hot = torch.nn.functional.one_hot(self.labels, num_classes=10).float()

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

def load_and_preprocess_data(csv_path):
    print(f"Lade Daten von {csv_path}...")
    df = pd.read_csv(csv_path)
    
    # Spec: First column label, remaining 784 are pixels
    labels = df.iloc[:, 0].values
    features = df.iloc[:, 1:].values
    
    # Spec: Split data into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(
        features, labels, test_size=0.15, random_state=42
    )
    
    train_dataset = MNISTCSVDataset(X_train, y_train)
    val_dataset = MNISTCSVDataset(X_val, y_val)
    
    return train_dataset, val_dataset

# ==========================================
# 2. Modell-Architektur
# ==========================================
class MNISTCNN(nn.Module):
    def __init__(self):
        super(MNISTCNN, self).__init__()
        # Convolutional Layers (CNN with 2 layers as suggested)
        # PyTorch erwartet intern (Batch, Channel, Height, Width).
        # Unser Input ist aber (Batch, Height, Width, Channel). 
        # Die Permutation findet in der forward-Methode statt.
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        
        # Fully Connected Layers
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Input Spec Check: (Batch, 28, 28, 1) -> Umwandeln für PyTorch Conv2D auf (Batch, 1, 28, 28)
        x = x.permute(0, 3, 1, 2)
        
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        logits = self.fc2(x) # Ausgabe sind Logits (für numerisch stabilere CrossEntropyLoss)
        return logits

# Wrapper-Klasse für den ONNX-Export, um Softmax einzubauen
class MNISTExportModel(nn.Module):
    def __init__(self, core_model):
        super(MNISTExportModel, self).__init__()
        self.core_model = core_model
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        # Spec: Apply softmax activation to convert logits to probabilities (1, 10)
        logits = self.core_model(x)
        return self.softmax(logits)

# ==========================================
# 3. Training & Validierung
# ==========================================
def train_and_evaluate(model, train_loader, val_loader, epochs=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    # CrossEntropyLoss funktioniert in PyTorch am besten mit Logits + One-Hot
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    print("Starte Training...")
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for inputs, labels_idx in train_loader:
            inputs, labels_idx = inputs.to(device), labels_idx.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs) # Outputs sind Logits
            loss = criterion(outputs, labels_idx)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
        # Validierung
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels_idx in val_loader:
                inputs, labels_idx = inputs.to(device), labels_idx.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels_idx.size(0)
                correct += (predicted == labels_idx).sum().item()
                
        accuracy = 100 * correct / total
        print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss/len(train_loader):.4f} | Validation Accuracy: {accuracy:.2f}%")
        
    return accuracy

# ==========================================
# 4. ONNX Export
# ==========================================
def export_to_onnx(model, filepath="mnist_model_v1.0.0.onnx"):
    model.eval()
    
    # Spec: Model Input Requirements -> Shape: (1, 28, 28, 1), dtype: float32
    dummy_input = torch.randn(1, 28, 28, 1, dtype=torch.float32)
    
    print(f"Exportiere Modell nach {filepath}...")
    torch.onnx.export(
        model, 
        dummy_input, 
        filepath,
        export_params=True,
        opset_version=14,          # Spec: Use standard PyTorch ONNX export settings
        do_constant_folding=True,
        input_names=['input'],     # Spec: Match what application expects
        output_names=['output'],   # Spec: Probabilities for 10 classes
        dynamic_axes={
            'input': {0: 'batch_size'},   # Erlaubt variable Batch-Größen (optional, aber Best Practice)
            'output': {0: 'batch_size'}
        }
    )
    print("ONNX-Export erfolgreich!")

# ==========================================
# Main Ausführung
# ==========================================
if __name__ == "__main__":
    # HINWEIS: Pfad zur CSV-Datei hier anpassen!
    csv_file_path = "Datasets/mnist/mnist_train.csv" 
    
    if not os.path.exists(csv_file_path):
        print(f"FEHLER: Die Datei '{csv_file_path}' wurde nicht gefunden.")
        print("Bitte lade einen MNIST-CSV-Datensatz herunter und passe den Pfad an.")
        exit(1)
        
    # 1. Daten laden
    train_data, val_data = load_and_preprocess_data(csv_file_path)
    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
    
    # 2. Modell initiieren
    core_model = MNISTCNN()
    
    # 3. Trainieren & Validieren
    final_accuracy = train_and_evaluate(core_model, train_loader, val_loader, epochs=5)
    
    # 4. Performance Check (Spec: Minimum accuracy 95%)
    if final_accuracy >= 95.0:
        print(f"\nErfolg: Die Modellgenauigkeit ({final_accuracy:.2f}%) übertrifft die geforderten 95%.")
        
        # 5. Export Wrapper erstellen (fügt Softmax am Ende hinzu)
        export_model = MNISTExportModel(core_model)
        export_model.to('cpu') # Export läuft standardmäßig am besten auf CPU
        
        # 6. Als ONNX exportieren
        export_to_onnx(export_model, filepath="mnist_model_v1.0.0.onnx")
    else:
        print(f"\nWARNUNG: Modell hat die Mindestgenauigkeit von 95% nicht erreicht ({final_accuracy:.2f}%).")
        print("Export wird abgebrochen. Bitte Epochen erhöhen oder Lernrate anpassen.")